In [1]:
from ipynb.fs.full. magic_flow_model import *
from ipynb.fs.full. data_loader import *
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import matplotlib.patches as mpatches
import copy
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, CosineAnnealingLR
import seaborn as sns

## Directory Setup

In [2]:
NUM_CLASSES = 7

base_path = os.getcwd()
print("Base path:", base_path)

# Define some additions
data_path = os.path.join(base_path, "datasets/scanner_data7")
model_folder = os.path.join(base_path, "checkpoints")
test_dir = os.path.join(data_path, "test")
pad_size = (184, 184) # pad to a dimension divisible by 4
batch_size = 16

test_dataset = GrayscaleImageDataset(
    root_dir=test_dir,
    num_classes=NUM_CLASSES,
    pad_to=pad_size,
    split="test"
)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Base path: /home/almalinux/data/jupyter_notebooks/cNF_folder/MAGIC-Flow/classification


## Model Initialization

In [5]:
filters_per_layer = [
    (16, 16, 32), (16, 16, 32), (16, 32, 32),  (16, 32, 32), (16, 32, 32),
    # 
    (16, 32, 32), (16, 32, 64), (16, 32, 64),
    # 
    (16, 32, 64), (16, 32, 64), (16, 64, 64),
    # 
    (32, 64, 64), (32, 64, 64), (32, 64, 64),
    # 
    (32, 64, 128), (32, 64, 128), (32, 64, 128),
    # 
    (64, 64, 128), (64, 64, 128), (64, 64, 128),
    # 
    (64, 128, 128), (64, 128, 128), (128, 256, 256), (128, 256, 256)
]


mask_types = (
    ['custom'] + ['checkerboard'] * 4 +                     
    (['channel'] * 3 + ['checkerboard'] * 3) * 2 +  
    ['channel'] * 3 +                          
    ['checkerboard'] * 4                      
)


real_nvp = CondMSRealNVP(
        num_coupling_layers=24,
        num_classes=NUM_CLASSES,
        input_shape=(1, 184, 184),
        squeeze_layers=[5, 11, 17],
        split_layers=[7, 13, 19],
        layer_filters=filters_per_layer,
        mask_types=mask_types,
        verbose = False
    )

# Assign to model the custom mask
mask_path = os.path.join(os.path.dirname(base_path), 'custom_png/coronal_slice_110_rotated.png') 
print(mask_path) 

# Load image as float tensor
mask_img = Image.open(mask_path).convert('L')
mask_np = np.array(mask_img)  # shape (H, W)
mask_tensor = torch.from_numpy(mask_np).float() / 255.0  # values 0.0 or 1.0
real_nvp.custom_mask_tensor = mask_tensor  # (H, W)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
real_nvp = real_nvp.to(device)


# Define the optimizer and scheduler
optimizer = torch.optim.Adam(real_nvp.parameters(), lr=1e-3)
n_epochs = 200
scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)

/home/almalinux/data/jupyter_notebooks/cNF_folder/MAGIC-Flow/custom_png/coronal_slice_110_rotated.png


## Load Checkpoint

In [6]:
# Load the checkpoint
checkpoint = torch.load(os.path.join(model_folder, "best_ckpt.pth"), map_location=device)

# Load model/optimizer/scheduler states
real_nvp.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

# Optionally resume training stats
start_epoch = checkpoint.get("epoch", 0)
global_step = checkpoint.get("global_step", 0)
history = checkpoint.get("history", {"loss": [], "val_loss": []})

print(f"Loaded model from epoch {start_epoch}, step {global_step}")

Loaded model from epoch 57, step 2394


## Evaluation - Classification Performance

In [ ]:
def compute_class_priors(dataset_root, class_names, device):
    class_counts = []
    total = 0

    for cls in class_names:
        cls_path = os.path.join(dataset_root, cls)
        count = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
        class_counts.append(count)
        total += count

    class_priors = torch.tensor([c / total for c in class_counts], device=device)
    return class_priors

def classify_images(model, data_loader, device, class_priors, temperature=600.0):
    """
    Classifica immagini usando log-likelihood del modello.
    
    Args:
        model: modello che ha la funzione predict(images, y)
        data_loader: DataLoader del dataset di test
        device: torch.device
        class_priors: tensor con prior delle classi [n_classes]
        temperature: float, per ammorbidire la softmax (default 1.0)
    
    Returns:
        preds: predizioni discrete (argmax)
        targets: etichette reali
        probs: probabilità continue per ogni classe
    """
    model.eval()
    all_preds, all_targets, all_probs = [], [], []

    log_priors = torch.log(class_priors).view(1, -1)  # [1, n_classes]

    with torch.no_grad():
        for images, one_hot in data_loader:
            images = images.to(device)
            one_hot = one_hot.to(device)
            B = images.size(0)

            targets = torch.argmax(one_hot, dim=1)  # indice della classe reale

            # --- Costruisci log-likelihood per ogni classe ---
            log_likelihoods = []
            for d in range(NUM_CLASSES):
                y = torch.zeros((B, NUM_CLASSES), device=device)
                y[:, d] = 1
                ll = model.predict(images, y)  # [B]
                log_likelihoods.append(ll.unsqueeze(-1))

            log_likelihoods = torch.cat(log_likelihoods, dim=1)  # [B, n_classes]

            # --- Aggiungi log priors ---
            log_posteriors = log_likelihoods + log_priors

            # --- Softmax numericamente stabile ---
            log_posteriors = log_posteriors - log_posteriors.max(dim=1, keepdim=True)[0]
            log_posteriors = log_posteriors / temperature
            probs = F.softmax(log_posteriors, dim=1)

            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_targets), np.array(all_probs)


def evaluate_and_visualize_with_age(model, test_loader, device, class_priors, class_names):

    preds, targets, probs = classify_images(model, test_loader, device, class_priors)

    # --- Confusion matrix + report
    cm = confusion_matrix(targets, preds)
    print("Classification Report:\n", classification_report(targets, preds, target_names=class_names, zero_division=0))
    print("Confusion Matrix:\n", cm)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

    # --- ROC AUC
    auc_score = roc_auc_score(targets, probs, multi_class="ovr")
    print("Overall ROC-AUC (OvR):", auc_score)
    
    from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score

    print("Balanced Accuracy:", balanced_accuracy_score(targets, preds))
    print("Cohen’s Kappa:", cohen_kappa_score(targets, preds))

    y_bin = label_binarize(targets, classes=list(range(len(class_names))))

    plt.figure(figsize=(7,6))
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.2f})")

    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (One-vs-Rest)")
    plt.legend()
    plt.show()



dataset_root = os.path.join(data_path, "train")
class_names = ['Guys', 'HH', 'IOP', 'Prisma', 'PrismaFit', 'SALD', 'Triotim']
class_priors = compute_class_priors(dataset_root, class_names, device)
evaluate_and_visualize_with_age(real_nvp, test_loader, device, class_priors, class_names)

## Likelihood Attribution Heatmaps

In [ ]:
# Metti il modello in modalità eval
real_nvp.eval()

# --- Smoothing con convoluzione Gaussiana ---
def gaussian_smooth_torch(heatmap, sigma=2):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    heatmap_tensor = torch.from_numpy(heatmap).float().to(device)
    size = int(2 * np.ceil(2 * sigma) + 1)
    x = torch.arange(size, device=device) - size // 2
    x_grid, y_grid = torch.meshgrid(x, x, indexing='ij')
    kernel = torch.exp(-(x_grid**2 + y_grid**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    kernel = kernel.view(1, 1, size, size)
    heatmap_tensor = heatmap_tensor.unsqueeze(0).unsqueeze(0)
    smooth = F.conv2d(heatmap_tensor, kernel, padding=size // 2)
    return smooth.squeeze().cpu().numpy()

# --- Estrazione immagine singola per classe ---
def get_single_image_for_class(test_loader, target_class, index_in_class=0):
    """
    Restituisce l'immagine 'index_in_class'-esima della classe target_class.
    """
    count = 0
    for x_batch, y_batch in test_loader:
        for i in range(len(y_batch)):
            if y_batch[i].argmax().item() == target_class:
                if count == index_in_class:
                    x_single = x_batch[i:i+1].to(real_nvp.device)
                    y_single = y_batch[i:i+1].to(real_nvp.device)
                    return x_single, y_single
                count += 1
    raise ValueError(f"Classe {target_class} non ha abbastanza immagini (index {index_in_class}).")


# --- Calcolo heatmap dal modello ---
def compute_heatmap(x_single, y_single):
    with torch.no_grad():
        log_det_dict = real_nvp.extract_log_det_maps(x_single, y_single)
        _, _, z_list = real_nvp(x_single, y_single, return_z_list=True)
        ll_map_z_i = [
            -0.5 * (z_i ** 2 + torch.log(torch.tensor(2 * torch.pi)))
            for z_i in z_list
        ]

        class Unsqueeze(torch.nn.Module):
            def forward(self, x):
                B, C, H, W = x.size()
                x = x.view(B, C // 4, 2, 2, H, W)
                x = x.permute(0, 1, 4, 2, 5, 3).contiguous()
                return x.view(B, C // 4, H * 2, W * 2)

        class MergeLayer(torch.nn.Module):
            def forward(self, x1, x2):
                return torch.cat([x1, x2], dim=1)

        unsqueeze = Unsqueeze()
        merge = MergeLayer()

        x = merge(
            ll_map_z_i[3] + ll_map_z_i[0] + sum(log_det_dict[(8, 23, 23)]),
            ll_map_z_i[3]
        )
        x = x + sum(log_det_dict[(16, 23, 23)])
        x = unsqueeze(x)
        x = x + sum(log_det_dict[(4, 46, 46)])
        x = merge(x, ll_map_z_i[2])
        x = x + sum(log_det_dict[(8, 46, 46)])
        x = unsqueeze(x)
        x = x + sum(log_det_dict[(2, 92, 92)])
        x = merge(x, ll_map_z_i[1])
        x = x + sum(log_det_dict[(4, 92, 92)])
        x = unsqueeze(x)
        x = x + sum(log_det_dict[(1, 184, 184)])
    return x.squeeze().cpu().numpy()

# --- Funzione per plottare con maschera esterna, NaN e legenda ---
def plot_and_save_heatmap(img_tensor, heatmap, target_class, mask_path=None,
                          save_dir="heatmaps", sigma=None, percentile=None):
    os.makedirs(save_dir, exist_ok=True)

    # Converte immagine in numpy
    img = img_tensor[0].cpu().numpy()
    if img.ndim == 3 and img.shape[0] == 3:
        img = img.transpose(1, 2, 0)
    elif img.ndim == 3 and img.shape[0] == 1:
        img = img.squeeze(0)

    # --- Carica maschera ---
    if mask_path is not None:
        mask_img = Image.open(mask_path).convert('L')
        mask_np = np.array(mask_img)
        mask_np = (mask_np > 127).astype(np.uint8)
    else:
        mask_np = np.ones_like(img, dtype=np.uint8)

    # Heatmap lisciata opzionalmente
    heatmap_plot = heatmap.copy()
    if sigma is not None:
        heatmap_plot = gaussian_smooth_torch(heatmap_plot, sigma=sigma)

    # Clip secondo percentili solo sul tessuto
    if percentile is not None:
        vmin, vmax = np.nanpercentile(heatmap_plot[mask_np == 1], percentile)
    else:
        vmin, vmax = np.nanmin(heatmap_plot[mask_np == 1]), np.nanmax(heatmap_plot[mask_np == 1])

    heatmap_clipped = np.clip(heatmap_plot, vmin, vmax)

    # Applica maschera: fuori maschera diventa NaN
    heatmap_masked = np.where(mask_np == 1, heatmap_clipped, np.nan)

    # --- Colormap con colore speciale per i NaN ---
    #cmap = copy.copy(mpl.cm.get_cmap("viridis"))
    #cmap.set_bad(color="black")  # colore per i NaN
    
    #cmap = copy.copy(mpl.cm.get_cmap("PiYG"))
    cmap = copy.copy(mpl.cm.get_cmap("RdBu_r"))
    #cmap = copy.copy(mpl.cm.get_cmap("PRGn"))
    cmap.set_bad(color="black")  # colore per i NaN

    # --- Plot ---
    plt.figure(figsize=(12, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05])

    ax0 = plt.subplot(gs[0])
    ax0.imshow(img, cmap='gray' if img.ndim == 2 else None)
    ax0.set_title(f"Classe: {target_class}")
    ax0.axis('off')

    ax1 = plt.subplot(gs[1])
    im = ax1.imshow(heatmap_masked, cmap=cmap, vmin=vmin, vmax=vmax)
    ax1.set_title("Heatmap log-likelihood (maschera PNG)")
    ax1.axis('off')

    # Colorbar
    from matplotlib.ticker import FormatStrFormatter

    cax = plt.subplot(gs[2])
    cbar = plt.colorbar(im, cax=cax)
    
    # Arrotonda alla prima cifra decimale
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    
    # Rende i tick labels in grassetto
    cbar.ax.tick_params(labelsize=13)  # dimensione font (puoi cambiarla)
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight('medium')

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"class_{target_class}_heatmap.pdf")
    plt.savefig(save_path, format='pdf', bbox_inches='tight')
    print(f"Heatmap classe {target_class} salvata in {save_path}")
    plt.show()

# --- Loop su tutte le classi ---
sigma = 1.8
percentile = [5, 95]
save_dir = os.path.join(base_path, 'attribution_maps')
os.makedirs(save_dir, exist_ok=True)  
mask_path = os.path.join(os.path.dirname(base_path), 'custom_png/seg_coronal_mask_padded.png')  

for target_class in range(NUM_CLASSES):
    x_single, y_single = get_single_image_for_class(test_loader, target_class, index_in_class=1)
    heatmap = compute_heatmap(x_single, y_single)
    plot_and_save_heatmap(
        x_single, heatmap, target_class,
        mask_path=mask_path,
        save_dir=save_dir, sigma=sigma, percentile=percentile
    )

In [ ]:
def get_images_for_classes(test_loader, class_indices):
    """
    Restituisce un dizionario {classe: (x_single, y_single)} dove per ogni classe
    si prende l'immagine con l'indice specificato in class_indices.

    Parameters
    ----------
    test_loader : DataLoader
        Il dataloader di test
    class_indices : dict
        Dizionario {classe: index_in_class}, dove index_in_class è l'immagine
        da prendere per quella classe

    Returns
    -------
    images_dict : dict
        {classe: (x_single, y_single)}
    """
    images_dict = {}
    
    # Per ogni classe di interesse, inizializza contatore
    class_counters = {cls: 0 for cls in class_indices.keys()}

    for x_batch, y_batch in test_loader:
        for i in range(len(y_batch)):
            cls = y_batch[i].argmax().item()
            if cls in class_indices:
                idx_target = class_indices[cls]
                count = class_counters[cls]

                if count == idx_target:
                    x_single = x_batch[i:i+1].to(real_nvp.device)
                    y_single = y_batch[i:i+1].to(real_nvp.device)
                    images_dict[cls] = (x_single, y_single)
                
                # Incrementa contatore
                class_counters[cls] += 1

        # Se abbiamo trovato tutte le immagini richieste, possiamo uscire
        if len(images_dict) == len(class_indices):
            break

    # Controllo finale
    missing = set(class_indices.keys()) - set(images_dict.keys())
    if missing:
        raise ValueError(f"Non ci sono abbastanza immagini per le classi: {missing}")

    return images_dict


class_indices = {0: 9, 1: 5, 2: 10, 3: 1, 4: 6, 5: 18, 6: 5}  
images_dict = get_images_for_classes(test_loader, class_indices)
sigma = 1.8
percentile = [5, 95]

for cls, (x_single, y_single) in images_dict.items():
    heatmap = compute_heatmap(x_single, y_single)
    plot_and_save_heatmap(
        x_single, heatmap, cls,
        mask_path=mask_path,
        save_dir=save_dir, sigma=sigma, percentile=percentile
    )